In [1]:
import os
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [7]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/postnauka.csv',
)

dataset.get_possible_modalities()

{'@2gramm', '@3gramm', '@author', '@post_tag', '@snippet', '@title', '@word'}

In [8]:
MAIN_MODALITY = '@word'

In [9]:
dataset._data.head()

,id,vw_text,raw_text
id,,,
1.txt,1.txt,1.txt |@author fuchs preobrazhensky tabachniko...,@title Автограф # «Математический дивертисмент...
2.txt,2.txt,2.txt |@word книга:2 лекция:3 рассматриваться:...,@title Главы: Маскулинности в российском конте...
3.txt,3.txt,3.txt |@word развитие появляться пиджина:4 бел...,@title Пиджины и креольские языки | @snippet Л...
4.txt,4.txt,4.txt |@word стандартный задача:3 состоять:4 р...,@title FAQ: Физиология микроводорослей | @snip...
5.txt,5.txt,5.txt |@2gramm повседневный_практика государст...,@title Русская государственная идеология | @sn...


In [10]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [11]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 4.69 s, sys: 242 ms, total: 4.93 s
Wall time: 4.87 s


In [12]:
co_occurences.shape

(19186, 19186)

In [13]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [14]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [15]:
KnownModel

<enum 'KnownModel'>

In [16]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [17]:
NUM_TOPICS = 50  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

## Test

In [18]:
PARAMS_EXPLORED[KnownModel.PLSA]


model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

In [25]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@word',
 'PerplexityScore@word',
 'TopicKernel@word.average_coherence',
 'TopicKernel@word.average_contrast',
 'TopicKernel@word.average_purity',
 'TopicKernel@word.average_size',
 'TopicKernel@word.coherence',
 'TopicKernel@word.contrast',
 'TopicKernel@word.purity',
 'TopicKernel@word.size',
 'TopicKernel@word.tokens']

In [35]:
model.scores['PerplexityScore@word']

[18851.2265625,
 5242.5166015625,
 4891.203125,
 4253.7138671875,
 3764.55810546875,
 3490.1767578125,
 3323.228515625,
 3216.050537109375,
 3144.248779296875,
 3094.3955078125,
 3058.91015625,
 3032.79150390625,
 3012.852783203125,
 2997.23876953125,
 2984.629150390625,
 2974.4453125,
 2966.25927734375,
 2959.608154296875,
 2954.128662109375,
 2949.429931640625]

In [32]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[0.7405018]
{0: array([1.14421437]), 1: array([0.50349005]), 2: array([0.82625659]), 3: array([0.9157611]), 4: array([0.63578375]), 5: array([0.48978686]), 6: array([0.70070286]), 7: array([0.80382441]), 8: array([0.74667564]), 9: array([0.5949052]), 10: array([0.8255863]), 11: array([0.68035692]), 12: array([0.65376446]), 13: array([0.88114531]), 14: array([0.79151262]), 15: array([0.84225123]), 16: array([0.91785754]), 17: array([0.65876494]), 18: array([0.43248484]), 19: array([0.76491107])}
Index(['topic_0', 'topic_1'], dtype='object')
diversity_euclidean
0.05009114090036071
Index(['topic_0', 'topic_1'], dtype='object')
diversity_jensenshannon
0.05009114090036071
Index(['topic_0', 'topic_1'], dtype='object')
diversity_hellinger
0.05009114090036071
Index(['topic_0', 'topic_1'], dtype='object')
diversity_cosine
0.05009114090036071


In [29]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality  token       
@word     материал        0.009624
          использовать    0.008002
          структура       0.007532
          технология      0.006426
          атом            0.006400
                            ...   
          стыд            0.000000
          малозаметный    0.000000
          english         0.000000
          cinema          0.000000
          аврелий         0.000000
Name: topic_18, Length: 19186, dtype: float32

In [64]:
model.class_ids

{'@word': 1}

In [30]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [17]:
MAIN_MODALITY

'@word'

In [18]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [19]:
BEST_PARAMS = dict()

## PLSA

In [20]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [21]:
NUM_TOPICS

50

In [43]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/artm/master_component.py:654: DeprecationWarning: invalid escape sequence \*
  """
/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/artm/master_component.py:760: DeprecationWarning: invalid escape sequence \d
  """


KeyboardInterrupt: 

In [22]:
BEST_PARAMS[KnownModel.PLSA] = None

## Sparse

In [27]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [39]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027

(-0.05, 0.1)
0


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314




In [41]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 3350.7021484375,
    'coherence_20': array([0.79300455]),
    'diversity_euclidean': 0.06043655558783355,
    'diversity_jensenshannon': 0.06043655558783355,
    'diversity_hellinger': 0.06043655558783355,
    'diversity_cosine': 0.06043655558783355},
   'topic_coherences': {0: 0.4613815887605477,
    1: 0.7254412703733861,
    2: 1.1825977284049987,
    3: 0.8628769522261815,
    4: 0.9995690399053021,
    5: 0.9733058107435728,
    6: 0.6980303292285863,
    7: 0.5761489198688887,
    8: 0.7483147931845104,
    9: 0.8621148798441607,
    10: 1.1127012218906336,
    11: 0.5671635792800231,
    12: 0.7538572393166724,
    13: 0.5220896659030285,
    14: 0.5394795375149565,
    15: 0.8817214328684023,
    16: 1.1619803113527283,
    17: 0.6217772221062762,
    18: 0.8775876214854835,
    19: 0.7319519205247833}}, {'scores': {'perplexity': 3328.0341796875,
    'coherence_20': array([0.78209502]),
    'diversity_euclidean': 0.05763263478401777,

In [42]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 3331.8850911458335
(-0.05, 0.1) 3491.5355631510415
(-0.1, 0.05) 3475.8790690104165
(-0.1, 0.1) 3637.692138671875


In [ ]:
# Best: (-0.05, 0.05) 3331.8850911458335

In [23]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

## Decorrelation

In [17]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [23]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [32]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [33]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.02, 0.05)
0
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02

(0.02, 0.1)
0


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01




In [34]:
len(results)

8

In [35]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.02, 0.05) 3080.7899576822915
(0.02, 0.1) 3209.5741373697915
(0.05, 0.05) 3085.8776041666665
(0.05, 0.1) 3216.2530110677085
(0.1, 0.05) 3148.940673828125
(0.1, 0.1) 3275.7158203125
(0.01, 0.05) 3081.0397135416665
(0.01, 0.1) 3209.85693359375


In [ ]:
# Best: (0.02, 0.05) 3080.7899576822915

In [24]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [27]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [28]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [29]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [36]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.02, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.02
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.02
2


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01





In [37]:
len(results)

16

In [38]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(0.02, -0.05, 0.05) 3341.3675130208335
(0.02, -0.05, 0.1) 3497.8352864583335
(0.02, -0.1, 0.05) 3486.6405436197915
(0.02, -0.1, 0.1) 3645.3518880208335
(0.05, -0.05, 0.05) 3367.4226888020835
(0.05, -0.05, 0.1) 3518.19677734375
(0.05, -0.1, 0.05) 3512.6874186197915
(0.05, -0.1, 0.1) 3666.226318359375
(0.1, -0.05, 0.05) 3442.3375651041665
(0.1, -0.05, 0.1) 3582.7361653645835
(0.1, -0.1, 0.05) 3589.3958333333335
(0.1, -0.1, 0.1) 3733.7527669270835
(0.01, -0.05, 0.05) 3335.7916666666665
(0.01, -0.05, 0.1) 3493.9044596354165
(0.01, -0.1, 0.05) 3480.2273763020835
(0.01, -0.1, 0.1) 3640.7910970052085


In [ ]:
# Best: (0.01, -0.05, 0.05) 3335.7916666666665

# Close: (0.02, -0.05, 0.05) 3341.3675130208335

In [25]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

## TLESS

In [19]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [20]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [22]:
results

[{'scores': {'perplexity': 3648.019287109375,
   'coherence_20': array([0.73268797]),
   'diversity_euclidean': 0.0828906236701155,
   'diversity_jensenshannon': 0.0828906236701155,
   'diversity_hellinger': 0.0828906236701155,
   'diversity_cosine': 0.0828906236701155},
  'topic_coherences': {0: 0.5748634448468164,
   1: 0.730467322022561,
   2: 0.4112179157799582,
   3: 1.008451239401693,
   4: 1.1553236136598417,
   5: 0.5280133176474108,
   6: 0.597173470430456,
   7: 0.4813047901914537,
   8: 0.8452729644551105,
   9: 1.1083068584696496,
   10: 1.0975959440604885,
   11: 0.5723046051892471,
   12: 0.8491850406145895,
   13: 0.5719212219989672,
   14: 0.4161534136724659,
   15: 0.9974945274670723,
   16: 1.3190569328602562,
   17: 0.4682329257973112,
   18: 0.5698427247201727,
   19: 0.35157720925244956}},
 {'scores': {'perplexity': 3635.931884765625,
   'coherence_20': array([0.71696431]),
   'diversity_euclidean': 0.08295491382261991,
   'diversity_jensenshannon': 0.0829549138226

In [ ]:
# Best:

In [26]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [41]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [42]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [43]:
results

{'symmetric': [{'scores': {'perplexity': 2994.8544921875,
    'coherence_20': array([0.72912617]),
    'diversity_euclidean': 0.04538816307890767,
    'diversity_jensenshannon': 0.04538816307890767,
    'diversity_hellinger': 0.04538816307890767,
    'diversity_cosine': 0.04538816307890767},
   'topic_coherences': {0: 0.4747744148677699,
    1: 0.7036357726509829,
    2: 0.6385750582879668,
    3: 0.8588277998967593,
    4: 0.8910872346278833,
    5: 0.8629812802644291,
    6: 0.6893087965800643,
    7: 0.5541064624279066,
    8: 0.8431573420660817,
    9: 0.922791744160493,
    10: 1.1066082291804975,
    11: 0.4860267452007646,
    12: 0.6664376516550651,
    13: 0.4953589831508357,
    14: 0.4460860341997396,
    15: 0.865571529527662,
    16: 1.0827362012402582,
    17: 0.6510241808664234,
    18: 0.7806403172775794,
    19: 0.5627876636567518}},
  {'scores': {'perplexity': 2964.62841796875,
    'coherence_20': array([0.742071]),
    'diversity_euclidean': 0.04512760284455174,
    

In [44]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

symmetric 2980.0504557291665
asymmetric 2977.8008626302085
heuristic 3156.46337890625


In [ ]:
# Best: asymmetric 2977.8008626302085

In [27]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'asymmetric',
}

In [37]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [28]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [27]:
BEST_PARAMS = {KnownModel.PLSA: None,
     KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
      'sparse_sp_tau': -0.05,
      'smooth_bcg_tau': 0.05},
     KnownModel.TLESS: None,
     KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
      'smooth_bcg_tau': 0.05},
     KnownModel.LDA: {'prior': 'asymmetric'}}

In [19]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [20]:
NUM_TRAINS = 20  # 100
COHERENCES = list()

In [21]:
SAVE_FOLDER = 'results50/postnauka'

! mkdir -p $SAVE_FOLDER

In [22]:
! ls

20_Newsgroups__internals
ARTM-Models-20NewsGroups-T20.ipynb
ARTM-Models-20NewsGroups-T50.ipynb
ARTM-Models-MKB10-T20-Copy1.ipynb
ARTM-Models-MKB10-T20.ipynb
ARTM-Models-MKB10-T50.ipynb
ARTM-Models-PostNauka-T20.ipynb
ARTM-Models-PostNauka-T50.ipynb
ARTM-Models-RTL-Wiki-Person-T20-Copy1.ipynb
ARTM-Models-RTL-Wiki-Person-T20.ipynb
ARTM-Models-RTL-Wiki-Person-T50-Copy1.ipynb
ARTM-Models-RTL-Wiki-Person-T50.ipynb
ARTM-Models-RuWikiGood-T20.ipynb
ARTM-Models-RuWikiGood-T50.ipynb
BERTopic
BERTopic-Coherence-20NewsGroups-T20.ipynb
BERTopic-Coherence-20NewsGroups-T50.ipynb
BERTopic-Coherence-MKB10-T20.ipynb
BERTopic-Coherence-MKB10-T50.ipynb
BERTopic-Coherence-PostNauka-T20.ipynb
BERTopic-Coherence-PostNauka-T50.ipynb
BERTopic-Coherence-RTL-Wiki-Person-T20.ipynb
BERTopic-Coherence-RTL-Wiki-Person-T50.ipynb
BERTopic-Coherence-RuWikiGood-T20.ipynb
BERTopic-Coherence-RuWikiGood-T50-2.ipynb
_BERTopic-Coherence-RuWikiGood-T50.ipynb
BERTopic-Coherence-RuWikiGood-T50.ipynb
Experiment_v2.ipynb
Experim

In [23]:
# PLSA

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [36]:
scores

[{'perplexity': 2386.130126953125,
  'coherence_20': array([0.85991201]),
  'diversity_euclidean': 0.05921866707644925,
  'diversity_jensenshannon': 0.636158734761917,
  'diversity_hellinger': 0.7354147980749794,
  'diversity_cosine': 0.7901747540652608},
 {'perplexity': 2399.628662109375,
  'coherence_20': array([0.82238304]),
  'diversity_euclidean': 0.054966288269833664,
  'diversity_jensenshannon': 0.6307451752277161,
  'diversity_hellinger': 0.7291875167915914,
  'diversity_cosine': 0.7697991221402324},
 {'perplexity': 2405.33837890625,
  'coherence_20': array([0.79394148]),
  'diversity_euclidean': 0.05829248934037062,
  'diversity_jensenshannon': 0.6313468456799338,
  'diversity_hellinger': 0.7296832693827333,
  'diversity_cosine': 0.7860506459406216},
 {'perplexity': 2396.16552734375,
  'coherence_20': array([0.821456]),
  'diversity_euclidean': 0.057263842223110574,
  'diversity_jensenshannon': 0.6345552133968381,
  'diversity_hellinger': 0.7337726816040979,
  'diversity_cosin

In [24]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [25]:
scores[0]

{'scores': {'perplexity': 2386.130126953125,
  'coherence_20': 0.8599120084020605,
  'diversity_euclidean': 0.05921866598644924,
  'diversity_jensenshannon': 0.6361587407904877,
  'diversity_hellinger': 0.7354148045266111,
  'diversity_cosine': 0.7901747650323747},
 'topic_coherences': {0: 1.078655325662046,
  1: 0.8502538318972109,
  2: 1.01033633217497,
  3: 1.031573822185498,
  4: 1.234329732423872,
  5: 1.0378604167125132,
  6: 0.7388875212118423,
  7: 0.6763906110353385,
  8: 1.2531593593046138,
  9: 0.9416753887989543,
  10: 1.303465929295518,
  11: 0.5713699269788047,
  12: 0.9386809940488807,
  13: 0.7931502951486508,
  14: 0.8537519927306221,
  15: 0.9710176013525604,
  16: 1.3287026567416074,
  17: 1.0134332640028447,
  18: 0.655309168285805,
  19: 0.6098772865520181,
  20: 0.9676619683545787,
  21: 1.1103637303457896,
  22: 0.5972853812881637,
  23: 0.5992637892663921,
  24: 0.6852913544376232,
  25: 1.1049875138809337,
  26: 1.0633524319374177,
  27: 0.8820828986558864,
  2

In [26]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [40]:
len(COHERENCES)

1000

In [41]:
COHERENCES[:10]

[1.078655325662046,
 0.8502538318972109,
 1.01033633217497,
 1.031573822185498,
 1.234329732423872,
 1.0378604167125132,
 0.7388875212118423,
 0.6763906110353385,
 1.2531593593046138,
 0.9416753887989543]

In [42]:
COHERENCES[:20]

[1.078655325662046,
 0.8502538318972109,
 1.01033633217497,
 1.031573822185498,
 1.234329732423872,
 1.0378604167125132,
 0.7388875212118423,
 0.6763906110353385,
 1.2531593593046138,
 0.9416753887989543,
 1.303465929295518,
 0.5713699269788047,
 0.9386809940488807,
 0.7931502951486508,
 0.8537519927306221,
 0.9710176013525604,
 1.3287026567416074,
 1.0134332640028447,
 0.655309168285805,
 0.6098772865520181]

In [28]:
# Sparse

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    COHERENCES.extend(
        list(results['topic_coherences'].values())
    )

0 smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.07116935463855091
sparse_theta_sp: -0.4011325611325611
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [29]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [30]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [46]:
len(COHERENCES)

2000

In [47]:
COHERENCES[-10:]

[1.3896209799652794,
 0.8401899540324315,
 0.9950089083854942,
 0.5330425176461779,
 1.006589789222807,
 0.7854359775141373,
 1.1658846121006372,
 0.5677920977658619,
 1.0009905136906516,
 0.6927262396687527]

In [48]:
max(COHERENCES)

1.8514334195866133

In [31]:
def train_many(model_family, save_file_path):
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        COHERENCES.extend(
            list(results['topic_coherences'].values())
        )

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [32]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [51]:
len(COHERENCES)

3000

In [33]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [53]:
len(COHERENCES)

4000

In [34]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.02
smooth_theta_0: 0.06572011113166809
smooth_theta_1: 0.057577431201934814
smooth_theta_2: 0.051230061799287796
smooth_theta_3: 0.04614320397377014
smooth_theta_4: 0.041975297033786774
smooth_theta_5: 0.03849795088171959
smooth_theta_6: 0.03555266931653023
smooth_theta_7: 0.033026017248630524
smooth_theta_8: 0.03083466738462448
smooth_theta_9: 0.028916023671627045
smooth_theta_10: 0.02722216211259365
smooth_theta_11: 0.025715766474604607
smooth_theta_12: 0.024367349222302437
smooth_theta_13: 0.02315329574048519
smooth_theta_14: 0.022054476663470268
smooth_theta_15: 0.021055227145552635
smooth_theta_16: 0.020142603665590286
smooth_theta_17: 0.01930580474436283
smooth_theta_18: 0.018535761162638664
smooth_theta_19: 0.017824791371822357
smooth_theta_20: 0.017166348174214363
smooth_theta_21: 0.01655481569468975
smooth_theta_22: 0.015985356643795967
smooth_theta_23: 0.015453769825398922
smooth_theta_24: 0.014956400729715824
smooth_theta_25: 0.01449005026370287
smooth_theta_

In [35]:
len(COHERENCES)

5000

In [56]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.4931439304293082
10: 0.5506208913207223
15: 0.5924331732699156
20: 0.6300154717967157
25: 0.665877212230463
30: 0.6945715103358118
35: 0.7281290701549838
40: 0.7615652924119195
45: 0.792218010231124
50: 0.8238610003128026
55: 0.8616097925735289
60: 0.8973339744138779
65: 0.9368914389465235
70: 0.9778323616394302
75: 1.0247260273224026
80: 1.0703095408390997
85: 1.1309678682056237
90: 1.2165637535266665
95: 1.3346302533916452


In [57]:
min(COHERENCES), max(COHERENCES)

(0.21497841490787317, 1.9140502220291464)

In [58]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(2549, 2404)

In [59]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.43542771669287395
98: 1.4438142090269461
